# 코드잇 스프린트 미션16: 이미지 모델 밴치마크 사이트 개발(추론 및 양자화)
---
여러분들은 지금까지 AI 스프린트 미션들을 수행하며 다양한 모델들을 학습 및 구현해보셨습니다. 
이번 미션에서는 그 모델들을 다시 가져와서, 여러 형태의 포맷으로 모델을 변환하여 저장해보는 실습을 해봅시다.




## 가이드라인
1. 이전 미션에서 다루었던 모델들 중 하나 이상을 자유롭게 선택하여 모델 학습을 진행합니다.
2. 아래의 3가지 타입의 모델로 변환하여 저장해봅시다.
    - `.pth` (PyTorch 기본 저장 형식)
    - `.pth` (양자화 된 버전)
    - `.onnx` (ONNX 형식)

## 데이터셋

`mnist data set`

- **데이터 구성**:
    - **학습**: 60,000장
    - **테스트**: 10,000장
    - **크기**: 28×28 grayscale
    - **클래스**: 0-9 숫자

**용량**: ~12MB (매우 작음)

## 사용 모델

`ViT`

# 학습 코드

## 1. 라이브러리

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: c:\Users\hambu\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

# 모델 클래스
from model import VisionTransformer, Config

In [14]:
class EvalConfig:
    """평가 설정"""
    data_root = './data'
    batch_size = 64
    num_workers = 2
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. 데이터 로드

In [15]:
def load_test_data(config):
    """MNIST 테스트 데이터셋 로드"""
    print("=" * 60)
    print("데이터 로드 중...")
    print("=" * 60)
    
    test_dataset = datasets.MNIST(
        root=config.data_root,
        train=False,
        download=True,
        transform=transforms.ToTensor()
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"✅ 테스트 데이터: {len(test_dataset)}개")
    print(f"✅ 배치 수: {len(test_loader)}")
    print(f"✅ 배치 크기: {config.batch_size}")
    
    return test_loader

## 3. 모델 정의 및 평가

In [16]:
def evaluate_model(model, test_loader, device):
    """모델 평가"""
    print("\n" + "=" * 60)
    print("모델 평가 시작")
    print("=" * 60)
    
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    start_time = time.time()
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"  [{batch_idx+1}/{len(test_loader)}] 처리 중...")
    
    eval_time = time.time() - start_time
    avg_loss = running_loss / len(test_loader)
    accuracy = 100. * correct / total
    
    print("\n" + "=" * 60)
    print("평가 완료!")
    print("=" * 60)
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"정답: {correct}/{total}")
    print(f"소요 시간: {eval_time:.1f}s")
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'time': eval_time
    }


## 메인 실행

In [17]:
def main():
    """평가 파이프라인 실행"""
    eval_config = EvalConfig()
    model_config = Config()
    
    print("=" * 60)
    print("Vision Transformer (ViT) - MNIST Evaluation")
    print("=" * 60)
    print(f"디바이스: {eval_config.device}")
    print(f"배치 크기: {eval_config.batch_size}")
    
    # 평가할 모델 경로 입력
    model_path = input("\n평가할 모델 경로: ")
    
    # 1. 데이터 로드
    test_loader = load_test_data(eval_config)
    
    # 2. 모델 생성 및 가중치 로드
    print("\n" + "=" * 60)
    print("모델 로드 중...")
    print("=" * 60)
    
    model = VisionTransformer(model_config).to(eval_config.device)
    
    try:
        weights = torch.load(model_path, map_location=eval_config.device)
        model.load_state_dict(weights)
        print(f"✅ 모델 로드 완료: {model_path}")
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {model_path}")
        return
    except Exception as e:
        print(f"❌ 모델 로드 실패: {e}")
        return
    
    # 모델 정보
    total_params = sum(p.numel() for p in model.parameters())
    print(f"총 파라미터: {total_params:,}")
    
    # 3. 평가 수행
    results = evaluate_model(model, test_loader, eval_config.device)
    
    # 4. 결과 요약
    print("\n" + "=" * 60)
    print("최종 결과")
    print("=" * 60)
    print(f"📁 모델: {model_path}")
    print(f"🎯 정확도: {results['accuracy']:.2f}%")
    print(f"📉 Loss: {results['loss']:.4f}")
    print(f"✅ 정답: {results['correct']}/{results['total']}")
    print(f"⏱️  소요 시간: {results['time']:.1f}s")

In [18]:
if __name__ == "__main__":
    main()

Vision Transformer (ViT) - MNIST Evaluation
디바이스: cpu
배치 크기: 64
데이터 로드 중...
✅ 테스트 데이터: 10000개
✅ 배치 수: 157
✅ 배치 크기: 64

모델 로드 중...
✅ 모델 로드 완료: ./models/mission_16_ViT_v1(epoch=10).pth
총 파라미터: 1,199,882

모델 평가 시작
  [50/157] 처리 중...
  [100/157] 처리 중...
  [150/157] 처리 중...

평가 완료!
Loss: 0.1906
Accuracy: 93.96%
정답: 9396/10000
소요 시간: 14.1s

최종 결과
📁 모델: ./models/mission_16_ViT_v1(epoch=10).pth
🎯 정확도: 93.96%
📉 Loss: 0.1906
✅ 정답: 9396/10000
⏱️  소요 시간: 14.1s
